In [53]:
from google.cloud import bigquery
from google.oauth2 import service_account

import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

import seaborn as sns
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [3]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [4]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [7]:
# job_config = bigquery.LoadJobConfig(
#   schema=[
#     bigquery.SchemaField(name='poi_index',field_type='NUMERIC'),
#     bigquery.SchemaField(name='lat',field_type='NUMERIC'),
#     bigquery.SchemaField(name='lon',field_type='NUMERIC')
#   ],
#   create_disposition="CREATE_IF_NEEDED",
#   write_disposition="WRITE_APPEND",
#   source_format=bigquery.SourceFormat.CSV,
#   field_delimiter='|'
# )

In [6]:
# # load table from file
# poi_df.to_csv(
#   'poi_df.csv',
#   header=False,
#   index=False,
#   sep='|'
# )
# load_job = client.load_table_from_file(
#   open('poi_df.csv','rb'),
#   'temp.poi_df',
#   job_config=job_config
# )
# load_job.result()

In [5]:
job = client.query(
  f'''
  select
    deal_amount,
    contract_date,
    floor_ floor,
    plottage,
    pnu,
    st_x(trade_case_point) longitude,
    st_y(trade_case_point) latitude,
    building_type,
    building_detail_type,
    building_use,
    build_year,
    building_area,
    sig_cd,
    emd_cd,
    land_use
  from m2.cremao_real_estate_trade_case
  where
    sido_cd = '11' and
    building_type = '집합' and
    contract_date >= '2020-01-01'
  '''
)
deal_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [6]:
deal_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12664 entries, 0 to 12663
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   deal_amount           12664 non-null  Int64  
 1   contract_date         12664 non-null  dbdate 
 2   floor                 12564 non-null  Int64  
 3   plottage              0 non-null      float64
 4   pnu                   12664 non-null  object 
 5   longitude             12664 non-null  float64
 6   latitude              12664 non-null  float64
 7   building_type         12664 non-null  object 
 8   building_detail_type  12664 non-null  object 
 9   building_use          12664 non-null  object 
 10  build_year            12623 non-null  object 
 11  building_area         12664 non-null  float64
 12  sig_cd                12664 non-null  object 
 13  emd_cd                12664 non-null  object 
 14  land_use              12664 non-null  object 
dtypes: Int64(2), dbdate

In [ ]:
# create columns from contract_date
deal_df['contract_year'] = [
  date.year
  for date
  in deal_df['contract_date']
]
deal_df['contract_month'] = [
  date.month
  for date
  in deal_df['contract_date']
]
deal_df['contract_day'] = [
  date.day
  for date
  in deal_df['contract_date']
]

In [36]:
deal_df['building_age'] = deal_df['contract_year'] - deal_df['build_year'].astype('float')

In [38]:
deal_df = deal_df[[
  'deal_amount','contract_year','contract_month','contract_day','floor','plottage','land_use','build_year','building_age','building_type','building_detail_type','building_use','building_area','sig_cd','emd_cd','pnu','longitude','latitude'
]]

In [ ]:
# # connect to localhost
# conn = psycopg2.connect(
#   host='localhost',
#   port=5432,
#   database='postgres',
#   user='postgres',
#   password='postgres'
# )
# conn.set_session(autocommit=True)
# cursor = conn.cursor()

In [80]:
deal_df_clean = deal_df[0:0]

for sig in deal_df['sig_cd'].unique():
  tmp_df = deal_df[deal_df['sig_cd'] == sig]
  # IQR 계산
  Q1 = tmp_df['deal_amount'].quantile(0.25)
  Q3 = tmp_df['deal_amount'].quantile(0.75)
  IQR = Q3 - Q1

  # # 이상치 제거
  deal_df_clean = pd.concat([
    deal_df_clean,
    tmp_df[~((tmp_df['deal_amount'] < (Q1 - 1.5 * IQR)) | (tmp_df['deal_amount'] > (Q3 + 1.5 * IQR)))]
  ])

In [81]:
deal_df_clean

,deal_amount,contract_year,contract_month,contract_day,floor,plottage,land_use,build_year,building_age,building_type,building_detail_type,building_use,building_area,sig_cd,emd_cd,pnu,longitude,latitude
1,1090000000,2021,4,20,4,NaN,일반상업,2009,12.0,집합,공장/창고,공장,120.53,11140,11140159,1114015900100420000,126.992519,37.563821
199,400000000,2021,12,13,1,NaN,중심상업,2008,13.0,집합,상가/사무실,제2종근린생활,10.53,11140,11140105,1114010500101990040,126.983868,37.564710
204,340000000,2022,4,7,2,NaN,일반상업,2021,1.0,집합,상가/사무실,제2종근린생활,46.55,11140,11140132,1114013200103340000,126.997519,37.562919
205,688500000,2022,4,7,1,NaN,일반상업,2021,1.0,집합,상가/사무실,숙박,43.55,11140,11140132,1114013200103340000,126.997519,37.562919
206,235000000,2022,3,5,13,NaN,일반상업,2005,17.0,집합,상가/사무실,숙박,43.34,11140,11140133,1114013300100220005,127.000742,37.563851
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12598,375000000,2021,1,11,23,NaN,일반상업,1999,22.0,집합,상가/사무실,업무,27.93,11680,11680118,1168011800104670006,127.051078,37.488120
12599,520000000,2020,6,29,27,NaN,일반상업,1999,21.0,집합,상가/사무실,업무,45.78,11680,11680118,1168011800104670006,127.051078,37.488120
12600,400000000,2020,6,4,1,NaN,일반상업,1998,22.0,집합,상가/사무실,제2종근린생활,15.22,11680,11680118,1168011800104670024,127.051883,37.486516
12601,700000000,2021,8,21,1,NaN,제3종일반주거,2006,15.0,집합,상가/사무실,제2종근린생활,61.56,11680,11680118,1168011800105270003,127.052652,37.495340


In [82]:
print(len(deal_df_clean), '/', len(deal_df))

11668 / 12664


In [39]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [75]:
try:
  deal_df_clean.to_sql(
    'm1_y',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

---
x 만들어 올리기